In [2]:
# =============================================================================
# unet_segmentation.py
# U-Net: Multi-class semantic segmentation on DeepFashion2 (top-5 categories)
#
# Dataset structure (same as Mask R-CNN notebook):
#   /kaggle/input/.../final_dataset/train/images/       <- .jpg
#   /kaggle/input/.../final_dataset/train/annotations/  <- .json
#   /kaggle/input/.../final_dataset/val/images/         <- .jpg
#   /kaggle/input/.../final_dataset/val/annotations/    <- .json
#
# Key difference from Mask R-CNN:
#   Mask R-CNN  → instance segmentation  [N binary masks per image]
#   U-Net       → semantic segmentation  [1 class map (H×W) per image]
#                 Each pixel is assigned one of 6 classes:
#                   0 = background
#                   1 = short sleeve top
#                   2 = trousers
#                   3 = shorts
#                   4 = long sleeve top
#                   5 = skirt
#
# Two training strategies:
#   Transfer Learning : ImageNet-pretrained ResNet-34 encoder
#   From Scratch      : randomly initialized encoder (baseline)
#
# Evaluation:
#   Per-class IoU, macro mIoU, Dice coefficient, pixel accuracy
#   Per-class ROC / AUC curves
#   F1, Precision, Recall (macro)
#
# NOTE: 33% subsampling — every other image used from both splits.
# =============================================================================


# =============================================================================
# SECTION 1: INSTALL DEPENDENCIES & IMPORTS
# =============================================================================

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# segmentation_models_pytorch gives us a clean U-Net with pretrained encoders
try:
    import segmentation_models_pytorch as smp
except ImportError:
    pip_install("segmentation-models-pytorch")
    import segmentation_models_pytorch as smp

import os
import json
import pickle
import random
import time
import copy
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms.functional as TF

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_curve, auc, average_precision_score,
)

print(f"PyTorch               : {torch.__version__}")
print(f"Torchvision           : {torchvision.__version__}")
print(f"segmentation-models   : {smp.__version__}")
print(f"CUDA available        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                   : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device                : {DEVICE}")


# =============================================================================
# SECTION 2: PATHS & HYPER-PARAMETERS
# =============================================================================

SRC_BASE  = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/pruned_deepfashion2/final_dataset"
SPLIT_PKL = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/split.pkl"

SRC_TRAIN_IMG = os.path.join(SRC_BASE, "train", "images")
SRC_TRAIN_ANN = os.path.join(SRC_BASE, "train", "annotations")
SRC_VAL_IMG   = os.path.join(SRC_BASE, "val",   "images")
SRC_VAL_ANN   = os.path.join(SRC_BASE, "val",   "annotations")

# taken standards, else val set se tune hokar ho jayega 

IMG_SIZE             = 512
BATCH_SIZE           = 8      # U-Net is lighter than Mask R-CNN; larger batch fine
NUM_EPOCHS_TRANSFER  = 5
NUM_EPOCHS_SCRATCH   = 5
LR_TRANSFER          = 1e-4
LR_SCRATCH           = 5e-4
WEIGHT_DECAY         = 1e-4
GRAD_CLIP            = 1.0

# Segmentation thresholds
MASK_THRESHOLD  = 0.5          # used when binarising per-class probability maps
IOU_THRESHOLD   = 0.5          # used in evaluation reporting
SCORE_THRESHOLD = 0.3          # minimum pixel confidence to assign a class

SUBSAMPLE_STEP = 3

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# SECTION 3: LABEL MAP & CATEGORY SETUP
# =============================================================================

# Same category→label mapping as the Mask R-CNN notebook.
# U-Net class indices:  0 = background, 1-5 = clothing classes
LABEL_MAP = {
    "short sleeve top": 0,
    "trousers":         1,
    "shorts":           2,
    "long sleeve top":  3,
    "skirt":            4,
}
top5         = {1, 8, 7, 2, 9}               # DeepFashion2 category_ids
cat_to_index = {1: 0, 8: 1, 7: 2, 2: 3, 9: 4}    # cat_id  → 0-based
index_to_cat = {0: "short sleeve top", 1: "trousers",
                2: "shorts",           3: "long sleeve top", 4: "skirt"}
cat_to_label = {cid: idx + 1 for cid, idx in cat_to_index.items()}  # cat_id → 1-based
label_to_idx = {lbl: lbl - 1 for lbl in cat_to_label.values()}       # 1-based → 0-based
num_classes  = 5              
num_labels   = num_classes + 1   


CLASS_NAMES = ["background"] + [index_to_cat[i] for i in range(num_classes)]

print("Semantic label mapping (0 = background):")
for cid, lbl in cat_to_label.items():
    print(f"  category_id={cid}  pixel_label={lbl}  ({index_to_cat[cat_to_index[cid]]})")

with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w") as f:
    json.dump(LABEL_MAP, f, indent=4)
print("Saved label_map.json")

PyTorch               : 2.10.0+cu128
Torchvision           : 0.25.0+cu128
segmentation-models   : 0.5.0
CUDA available        : True
GPU                   : Tesla T4
Device                : cuda
Semantic label mapping (0 = background):
  category_id=1  pixel_label=1  (short sleeve top)
  category_id=8  pixel_label=2  (trousers)
  category_id=7  pixel_label=3  (shorts)
  category_id=2  pixel_label=4  (long sleeve top)
  category_id=9  pixel_label=5  (skirt)
Saved label_map.json


In [3]:

# =============================================================================
# SECTION 4: 80/20 SPLIT + 50% SUBSAMPLING
# (identical logic to Mask R-CNN notebook — reuses split.pkl when available)
# =============================================================================

def split_train_folder(val_size=0.20, seed=42):
    valid_files, strat_labels = [], []
    ann_files = sorted(os.listdir(SRC_TRAIN_ANN))
    print(f"Scanning {len(ann_files)} annotation files...")

    for ann_file in ann_files:
        if not ann_file.endswith(".json"):
            continue
        img_file = ann_file.replace(".json", ".jpg")
        if not os.path.exists(os.path.join(SRC_TRAIN_IMG, img_file)):
            continue

        with open(os.path.join(SRC_TRAIN_ANN, ann_file)) as f:
            data = json.load(f)

        cats = [v["category_id"] for v in data.values()
                if isinstance(v, dict) and v.get("category_id") in top5]
        if not cats:
            continue

        valid_files.append(img_file)
        strat_labels.append(max(set(cats), key=cats.count))

    print(f"Valid images (full): {len(valid_files)}")
    tr, vl = train_test_split(
        valid_files, test_size=val_size, stratify=strat_labels, random_state=seed
    )
    print(f"  Train (80%): {len(tr)}  |  Val (20%): {len(vl)}")
    return tr, vl


if os.path.exists(SPLIT_PKL):
    with open(SPLIT_PKL, "rb") as f:
        saved = pickle.load(f)
    train_files_full = saved["train_files"]
    val_files_full   = saved["val_files"]
    print(f"Loaded split.pkl — {len(train_files_full)} train, {len(val_files_full)} val")
else:
    print("split.pkl not found — recomputing split...")
    train_files_full, val_files_full = split_train_folder()
    with open(SPLIT_PKL, "wb") as f:
        pickle.dump({"train_files": train_files_full, "val_files": val_files_full}, f)
    print("Saved new split.pkl")

train_files = train_files_full[::SUBSAMPLE_STEP]
val_files   = val_files_full[::SUBSAMPLE_STEP]

print(f"\n50% subsampling (step={SUBSAMPLE_STEP}):")
print(f"  train : {len(train_files_full):6d}  →  {len(train_files):6d} images")
print(f"  val   : {len(val_files_full):6d}  →  {len(val_files):6d} images")

val_eval_files_full = sorted([f for f in os.listdir(SRC_VAL_IMG) if f.endswith(".jpg")])
val_eval_files      = val_eval_files_full[::SUBSAMPLE_STEP]
print(f"  eval  : {len(val_eval_files_full):6d}  →  {len(val_eval_files):6d} images")

Loaded split.pkl — 115339 train, 28835 val

50% subsampling (step=3):
  train : 115339  →   38447 images
  val   :  28835  →    9612 images
  eval  :  23741  →    7914 images


In [4]:

# =============================================================================
# SECTION 5: CLASS WEIGHTS FOR LOSS FUNCTION
#
# Because background dominates the pixel space, we compute inverse-frequency
# weights per class and feed them to CrossEntropyLoss.
# We count pixels as "belonging to class c" if any annotation in that image
# covers that class — a fast proxy for true pixel counts.
# =============================================================================

def compute_class_weights(file_list, ann_folder):
    """
    Returns a FloatTensor of length num_labels (6) with inverse-frequency
    class weights for CrossEntropyLoss.  Background weight is set to 0.1
    so the model focuses on foreground classes.
    """
    class_counts = np.zeros(num_labels, dtype=float)
    class_counts[0] = len(file_list)   # every image has background

    for img_file in file_list:
        ann_path = os.path.join(ann_folder, img_file.replace(".jpg", ".json"))
        with open(ann_path) as f:
            data = json.load(f)
        seen = set()
        for v in data.values():
            if isinstance(v, dict) and v.get("category_id") in top5:
                lbl = cat_to_label[v["category_id"]]
                if lbl not in seen:
                    class_counts[lbl] += 1
                    seen.add(lbl)

    weights = 1.0 / (class_counts + 1e-6)
    weights[0] *= 0.1              # downweight background
    weights = weights / weights.sum() * num_labels   # normalise to sum=num_labels

    print("\nClass weights for CrossEntropyLoss:")
    for i, (name, w) in enumerate(zip(CLASS_NAMES, weights)):
        print(f"  [{i}] {name:25s}  count={int(class_counts[i]):6d}  weight={w:.4f}")
    return torch.tensor(weights, dtype=torch.float32)


class_weights = compute_class_weights(train_files, SRC_TRAIN_ANN)




# =============================================================================
# SECTION 6: OVERSAMPLING WEIGHTS (WeightedRandomSampler)
# Same logic as Mask R-CNN — ensures rare classes appear frequently.
# =============================================================================

def compute_sample_weights(file_list, ann_folder):
    class_counts = np.zeros(num_classes, dtype=float)
    for img_file in file_list:
        ann_path = os.path.join(ann_folder, img_file.replace(".jpg", ".json"))
        with open(ann_path) as f:
            data = json.load(f)
        for v in data.values():
            if isinstance(v, dict) and v.get("category_id") in top5:
                class_counts[cat_to_index[v["category_id"]]] += 1

    class_weights_arr = 1.0 / (class_counts + 1e-6)
    class_weights_arr /= class_weights_arr.sum()

    sample_weights = []
    for img_file in file_list:
        ann_path = os.path.join(ann_folder, img_file.replace(".jpg", ".json"))
        with open(ann_path) as f:
            data = json.load(f)
        w = 0.0
        for v in data.values():
            if isinstance(v, dict) and v.get("category_id") in top5:
                w = max(w, class_weights_arr[cat_to_index[v["category_id"]]])
        sample_weights.append(w if w > 0 else 1e-6)

    print("\nClass instance counts (subsampled train):")
    for i in range(num_classes):
        print(f"  [{i}] {index_to_cat[i]:25s}  count={int(class_counts[i]):6d}")
    return np.array(sample_weights, dtype=float)


sample_weights = compute_sample_weights(train_files, SRC_TRAIN_ANN)




Class weights for CrossEntropyLoss:
  [0] background                 count= 38447  weight=0.0343
  [1] short sleeve top           count= 18806  weight=0.7004
  [2] trousers                   count= 14619  weight=0.9010
  [3] shorts                     count=  9752  weight=1.3506
  [4] long sleeve top            count=  9422  weight=1.3979
  [5] skirt                      count=  8151  weight=1.6159

Class instance counts (subsampled train):
  [0] short sleeve top           count= 19075
  [1] trousers                   count= 14737
  [2] shorts                     count=  9831
  [3] long sleeve top            count=  9514
  [4] skirt                      count=  8215


In [14]:

# =============================================================================
# SECTION 7: HELPER — POLYGON → BINARY MASK
# (unchanged from Mask R-CNN notebook)
# =============================================================================

def polygons_to_mask(polygons, height, width):
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)
    for poly in polygons:
        if len(poly) < 6:
            continue
        coords = list(zip(poly[0::2], poly[1::2]))
        draw.polygon(coords, fill=1)
    return np.array(mask, dtype=np.uint8)


# =============================================================================
# SECTION 8: PYTORCH DATASET
#
# KEY CHANGE vs Mask R-CNN:
#   __getitem__ now returns (image_tensor, semantic_mask) where
#   semantic_mask is LongTensor [H, W] with pixel values 0..5.
#
#   All instance masks for the same image are merged into one map.
#   If two instances overlap, the last-written label wins (rare in fashion).
# =============================================================================

class DeepFashion2SemanticDataset(Dataset):
    """
    Returns:
        image        : FloatTensor [3, H, W]   values in [0,1]
        sem_mask     : LongTensor  [H, W]      pixel labels 0-5
    """

    def __init__(self, file_list, img_folder, ann_folder,
                 augment=False, img_size=IMG_SIZE):
        self.file_list  = list(file_list)
        self.img_folder = img_folder
        self.ann_folder = ann_folder
        self.augment    = augment
        self.img_size   = img_size

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_file = self.file_list[idx]
        img_path = os.path.join(self.img_folder, img_file)
        ann_path = os.path.join(self.ann_folder, img_file.replace(".jpg", ".json"))

        # ── load & resize image ────────────────────────────────────────────
        img      = Image.open(img_path).convert("RGB")
        orig_w, orig_h = img.size
        img      = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img_tensor = TF.to_tensor(img)          # [3, H, W]

        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h

        # ── build semantic mask (H×W, dtype=long, values 0..num_classes) ──
        sem_mask = np.zeros((self.img_size, self.img_size), dtype=np.int64)

        with open(ann_path) as f:
            data = json.load(f)

        for v in data.values():
            if not isinstance(v, dict):
                continue
            cat_id = v.get("category_id")
            if cat_id not in top5:
                continue

            pixel_label = cat_to_label[cat_id]   # 1..5

            segs = v.get("segmentation", [])
            scaled_segs = []
            for poly in segs:
                scaled = []
                for i, coord in enumerate(poly):
                    scaled.append(coord * scale_x if i % 2 == 0
                                  else coord * scale_y)
                scaled_segs.append(scaled)

            inst_mask = polygons_to_mask(scaled_segs, self.img_size, self.img_size)
            # write label into semantic map (later instances overwrite earlier ones)
            sem_mask[inst_mask == 1] = pixel_label

        sem_mask_t = torch.tensor(sem_mask, dtype=torch.long)   # [H, W]

        # ── augmentation ───────────────────────────────────────────────────
        if self.augment:
            img_tensor, sem_mask_t = self._augment(img_tensor, sem_mask_t)

        return img_tensor, sem_mask_t

    # ── augmentation helper ────────────────────────────────────────────────
    @staticmethod
    def _augment(image, sem_mask):
        """
        Geometric augmentations applied identically to image AND mask.
        Colour jitter applied to image only.
        """
        # Random horizontal flip
        if random.random() > 0.5:
            image    = TF.hflip(image)
            sem_mask = sem_mask.flip(-1)     # flip last dim (W)

        # Random vertical flip (mild — clothing is usually upright)
        if random.random() > 0.8:
            image    = TF.vflip(image)
            sem_mask = sem_mask.flip(-2)     # flip second-to-last dim (H)

        # Colour jitter (image only)
        image = TF.adjust_brightness(image, 1.0 + random.uniform(-0.2, 0.2))
        image = TF.adjust_contrast(image,   1.0 + random.uniform(-0.2, 0.2))
        image = TF.adjust_saturation(image, 1.0 + random.uniform(-0.1, 0.1))
        image = TF.adjust_hue(image,              random.uniform(-0.05, 0.05))

        return image, sem_mask


In [15]:


# NOTE: no custom collate_fn needed — images stack to [B,3,H,W],
#       masks stack to [B,H,W], both regular tensors.

# =============================================================================
# SECTION 9: DATALOADERS
# =============================================================================

print("\nBuilding DataLoaders...")

train_sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(train_files),
    replacement=True
)

train_dataset    = DeepFashion2SemanticDataset(
    train_files, SRC_TRAIN_IMG, SRC_TRAIN_ANN, augment=True)
val_fit_dataset  = DeepFashion2SemanticDataset(
    val_files, SRC_TRAIN_IMG, SRC_TRAIN_ANN, augment=False)
val_eval_dataset = DeepFashion2SemanticDataset(
    val_eval_files, SRC_VAL_IMG, SRC_VAL_ANN, augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=2, pin_memory=True
)
val_fit_loader = DataLoader(
    val_fit_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
val_eval_loader = DataLoader(
    val_eval_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

print(f"  train_loader    : {len(train_loader)} batches  ({len(train_dataset)} images)")
print(f"  val_fit_loader  : {len(val_fit_loader)} batches  ({len(val_fit_dataset)} images)")
print(f"  val_eval_loader : {len(val_eval_loader)} batches  ({len(val_eval_dataset)} images)")


# =============================================================================
# SECTION 10: DATASET DISTRIBUTION VISUALIZATION
# =============================================================================

def plot_dataset_distribution():
    def count_instances(file_list, ann_folder):
        counts = np.zeros(num_classes, dtype=int)
        for f in file_list:
            ann = os.path.join(ann_folder, f.replace(".jpg", ".json"))
            with open(ann) as fp:
                data = json.load(fp)
            for v in data.values():
                if isinstance(v, dict) and v.get("category_id") in top5:
                    counts[cat_to_index[v["category_id"]]] += 1
        return counts

    print("Counting instance distribution...")
    tr_cnt = count_instances(train_files,    SRC_TRAIN_ANN)
    vf_cnt = count_instances(val_files,      SRC_TRAIN_ANN)
    ve_cnt = count_instances(val_eval_files, SRC_VAL_ANN)

    x      = np.arange(num_classes)
    w      = 0.28
    labels = [index_to_cat[i] for i in range(num_classes)]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w, tr_cnt, w, label="Train 50% subsample", color="#4e79a7")
    ax.bar(x,     vf_cnt, w, label="Val 50% subsample",   color="#f28e2b")
    ax.bar(x + w, ve_cnt, w, label="Eval Val 50%",        color="#59a14f")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel("Instance count")
    ax.set_title("Instance Distribution — 50% Subsampled Splits (top-5 categories)")
    ax.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "unet_split_distribution.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")

plot_dataset_distribution()


# =============================================================================
# SECTION 11: AUGMENTATION VISUALIZATION
# =============================================================================

def visualize_augmentations():
    colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]
    samples = {}
    for img_file in train_files:
        ann_path = os.path.join(SRC_TRAIN_ANN, img_file.replace(".jpg", ".json"))
        with open(ann_path) as f:
            data = json.load(f)
        for v in data.values():
            if isinstance(v, dict) and v.get("category_id") in top5:
                idx = cat_to_index[v["category_id"]]
                if idx not in samples:
                    samples[idx] = img_file
        if len(samples) == num_classes:
            break

    fig, axes = plt.subplots(num_classes, 4,
                              figsize=(16, num_classes * 3.5))

    for row, (cat_idx, img_file) in enumerate(sorted(samples.items())):
        img   = Image.open(os.path.join(SRC_TRAIN_IMG, img_file)).convert("RGB")
        img   = img.resize((IMG_SIZE, IMG_SIZE))
        img_t = TF.to_tensor(img)

        sem_mask = torch.zeros(IMG_SIZE, IMG_SIZE, dtype=torch.long)

        axes[row][0].imshow(img)
        axes[row][0].set_title(f"{index_to_cat[cat_idx]}\nOriginal", fontsize=8)
        axes[row][0].axis("off")

        for col in range(1, 4):
            aug_t, _ = DeepFashion2SemanticDataset._augment(
                img_t.clone(), sem_mask.clone())
            aug_np = (aug_t.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
            axes[row][col].imshow(aug_np)
            axes[row][col].set_title(f"Augmented #{col}", fontsize=8)
            axes[row][col].axis("off")

    plt.suptitle("Data Augmentation Preview (per category)", fontsize=13)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "unet_augmentation_preview.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")

visualize_augmentations()


Building DataLoaders...
  train_loader    : 4806 batches  (38447 images)
  val_fit_loader  : 1202 batches  (9612 images)
  val_eval_loader : 990 batches  (7914 images)
Counting instance distribution...
Saved /kaggle/working/unet_split_distribution.png
Saved /kaggle/working/unet_augmentation_preview.png


In [17]:



# =============================================================================
# SECTION 12: SANITY CHECK — VISUALIZE ONE BATCH
# Shows image + colour-coded semantic mask side by side.
# =============================================================================

def visualize_batch(loader, title="Batch Sample",
                    save_name="unet_batch_sample.png", n_images=4):
    images, masks = next(iter(loader))

    palette = np.array([
        [0,   0,   0  ],   # 0 background  — black
        [228, 26,  28 ],   # 1 short sleeve — red
        [55,  126, 184],   # 2 trousers     — blue
        [77,  175, 74 ],   # 3 shorts       — green
        [152, 78,  163],   # 4 long sleeve  — purple
        [255, 127, 0  ],   # 5 skirt        — orange
    ], dtype=np.uint8)

    n = min(n_images, len(images))
    fig, axes = plt.subplots(n, 2, figsize=(10, n * 4))
    if n == 1:
        axes = [axes]

    for i in range(n):
        img_np  = (images[i].permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
        mask_np = masks[i].numpy().astype(np.int64)
        color_mask = palette[mask_np]           # [H, W, 3]

        axes[i][0].imshow(img_np)
        axes[i][0].set_title("Image", fontsize=9)
        axes[i][0].axis("off")

        axes[i][1].imshow(color_mask)
        axes[i][1].set_title("Ground-Truth Semantic Mask", fontsize=9)
        axes[i][1].axis("off")

    legend = [mpatches.Patch(color=palette[i] / 255., label=CLASS_NAMES[i])
              for i in range(num_labels)]
    fig.legend(handles=legend, loc="lower center",
               ncol=num_labels, fontsize=8)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")

print("\nVisualizing a training batch...")
visualize_batch(train_loader,
                title="Training Batch — Ground-Truth Semantic Masks (33% subsample)",
                save_name="unet_train_batch_sample.png")



Visualizing a training batch...
Saved /kaggle/working/unet_train_batch_sample.png


In [24]:
# =============================================================================
# SECTION 13: MODEL  —  U-Net with ResNet-34 encoder
# =============================================================================

def build_unet(pretrained=True):
    model = smp.Unet(
        encoder_name    = "resnet34",
        encoder_weights = "imagenet" if pretrained else None,
        in_channels     = 3,
        classes         = num_labels,    # 6
        activation      = None,          # raw logits
    )
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params    : {total:,}")
    print(f"  Trainable params: {trainable:,}")
    return model.to(DEVICE)


# =============================================================================
# SECTION 14: LOSS  —  0.5 × CrossEntropy  +  0.5 × Soft Dice
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        C     = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        oh    = F.one_hot(targets, C).permute(0,3,1,2).float()
        dims  = (0, 2, 3)
        inter = (probs * oh).sum(dims)
        union = probs.sum(dims) + oh.sum(dims)
        return 1.0 - ((2*inter + self.smooth) / (union + self.smooth)).mean()


class SegLoss(nn.Module):
    def __init__(self, weights):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(weight=weights.to(DEVICE))
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return 0.5 * self.ce(logits, targets) + 0.5 * self.dice(logits, targets)


# =============================================================================
# SECTION 15: METRICS
# =============================================================================

def iou_per_class(pred, true, n=num_labels):
    """numpy [H,W] → [n] array, NaN where class absent in both pred+true."""
    out = np.full(n, np.nan)
    for c in range(n):
        pc = pred == c; tc = true == c
        inter = (pc & tc).sum(); union = (pc | tc).sum()
        if union > 0:
            out[c] = inter / union
    return out


def dice_per_class(pred, true, n=num_labels, smooth=1.0):
    out = np.full(n, np.nan)
    for c in range(n):
        pc = (pred == c).astype(float); tc = (true == c).astype(float)
        inter = (pc * tc).sum(); denom = pc.sum() + tc.sum()
        if denom > 0:
            out[c] = (2*inter + smooth) / (denom + smooth)
    return out


def pixel_acc(pred, true):
    return (pred == true).mean()


# =============================================================================
# SECTION 16: TRAINING LOOP
# =============================================================================

def train_one_epoch(model, loader, optimizer, criterion, epoch, total):
    model.train()
    running = 0.0
    t0 = time.time()
    start_time = time.time()
    
    for i, (imgs, masks) in enumerate(loader):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        if GRAD_CLIP > 0:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        running += loss.item()
        if (i+1) % 50 == 0 or i == 0:
            elapsed = time.time() - start_time
            avg_time = elapsed / (i + 1)
            remaining = avg_time * (len(loader) - i - 1)
        
            print(
                f"  [{epoch}/{total}] batch {i+1}/{len(loader)} "
                f"loss={loss.item():.4f} "
                f"| {avg_time:.2f}s/batch "
                f"| ETA {remaining/60:.1f} min"
            )



    
    avg = running / len(loader)
    print(f"  → epoch {epoch}  {time.time()-t0:.0f}s  avg_loss={avg:.4f}")
    return avg


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_iou, all_dice, all_acc = [], [], []
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(1)
        for p, g in zip(preds.cpu().numpy(), masks.cpu().numpy()):
            all_iou.append(iou_per_class(p, g))
            all_dice.append(dice_per_class(p, g))
            all_acc.append(pixel_acc(p, g))
    iou_arr  = np.array(all_iou)
    dice_arr = np.array(all_dice)
    return {
        "loss":      total_loss / len(loader),
        "miou":      float(np.nanmean(iou_arr[:, 1:])),   # exclude bg
        "dice":      float(np.nanmean(dice_arr[:, 1:])),
        "pixel_acc": float(np.mean(all_acc)),
        "iou_cls":   np.nanmean(iou_arr,  axis=0),
        "dice_cls":  np.nanmean(dice_arr, axis=0),
    }


def run_training(model, num_epochs, lr, run_name):
    criterion = SegLoss(class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=lr / 50)

    history    = defaultdict(list)
    best_miou  = -1.0
    best_state = None

    for epoch in range(1, num_epochs + 1):
        

        
        print(f"\n{'='*55}\n[{run_name}] Epoch {epoch}/{num_epochs}\n{'='*55}")


        
        tr_loss = train_one_epoch(model, train_loader, optimizer,
                                  criterion, epoch, num_epochs)
        val_met = validate(model, val_fit_loader, criterion)
        scheduler.step()
        lr_now  = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_met["loss"])
        history["miou"].append(val_met["miou"])
        history["dice"].append(val_met["dice"])
        history["pixel_acc"].append(val_met["pixel_acc"])
        history["lr"].append(lr_now)

        print(f"  val_loss={val_met['loss']:.4f}  mIoU={val_met['miou']:.4f}  "
              f"Dice={val_met['dice']:.4f}  PixAcc={val_met['pixel_acc']:.4f}  "
              f"lr={lr_now:.2e}")
        
        for i, name in enumerate(CLASS_NAMES):
            v = val_met["iou_cls"][i]
            print(f"    [{i}] {name:25s}  IoU="
                  + (f"{v:.4f}" if not np.isnan(v) else "N/A"))

        if val_met["miou"] > best_miou:
            best_miou  = val_met["miou"]
            best_state = copy.deepcopy(model.state_dict())
            ckpt = os.path.join(OUTPUT_DIR, f"unet_{run_name}_best.pth")
            torch.save(best_state, ckpt)
            print(f"  best mIoU={best_miou:.4f}  saved → {ckpt}")

    model.load_state_dict(best_state)
    print(f"\n[{run_name}] done.  Best mIoU = {best_miou:.4f}")
    return model, dict(history)

In [25]:

# =============================================================================
# SECTION 17: TRAIN — TRANSFER LEARNING
# =============================================================================

print("\n" + "="*60)
print("TRANSFER LEARNING  (ImageNet ResNet-34 encoder)")
print("="*60)
model_transfer = build_unet(pretrained=True)
model_transfer, hist_transfer = run_training(
    model_transfer, NUM_EPOCHS_TRANSFER, LR_TRANSFER, "transfer")


TRANSFER LEARNING  (ImageNet ResNet-34 encoder)
  Total params    : 24,437,094
  Trainable params: 24,437,094

[transfer] Epoch 1/5
  [1/5] batch 1/4806 loss=1.4498 | 1.43s/batch | ETA 114.2 min
  [1/5] batch 50/4806 loss=1.3617 | 0.49s/batch | ETA 38.5 min
  [1/5] batch 100/4806 loss=1.0616 | 0.48s/batch | ETA 38.0 min
  [1/5] batch 150/4806 loss=1.0623 | 0.49s/batch | ETA 37.8 min
  [1/5] batch 200/4806 loss=1.0607 | 0.48s/batch | ETA 37.2 min
  [1/5] batch 250/4806 loss=1.1151 | 0.48s/batch | ETA 36.7 min
  [1/5] batch 300/4806 loss=1.0181 | 0.48s/batch | ETA 36.3 min
  [1/5] batch 350/4806 loss=1.1748 | 0.48s/batch | ETA 35.8 min
  [1/5] batch 400/4806 loss=1.0424 | 0.48s/batch | ETA 35.4 min
  [1/5] batch 450/4806 loss=1.0208 | 0.48s/batch | ETA 35.0 min
  [1/5] batch 500/4806 loss=1.0371 | 0.48s/batch | ETA 34.6 min
  [1/5] batch 550/4806 loss=0.9140 | 0.48s/batch | ETA 34.2 min
  [1/5] batch 600/4806 loss=0.8269 | 0.48s/batch | ETA 33.8 min
  [1/5] batch 650/4806 loss=0.8560 | 

In [26]:


# =============================================================================
# SECTION 18: TRAIN — FROM SCRATCH
# =============================================================================

print("\n" + "="*60)
print("FROM SCRATCH  (random init)")
print("="*60)
model_scratch = build_unet(pretrained=False)
model_scratch, hist_scratch = run_training(
    model_scratch, NUM_EPOCHS_SCRATCH, LR_SCRATCH, "scratch")


FROM SCRATCH  (random init)
  Total params    : 24,437,094
  Trainable params: 24,437,094

[scratch] Epoch 1/5
  [1/5] batch 1/4806 loss=1.5225 | 1.38s/batch | ETA 110.4 min
  [1/5] batch 50/4806 loss=1.3439 | 0.49s/batch | ETA 39.1 min
  [1/5] batch 100/4806 loss=1.2981 | 0.49s/batch | ETA 38.8 min
  [1/5] batch 150/4806 loss=1.3466 | 0.49s/batch | ETA 38.0 min
  [1/5] batch 200/4806 loss=1.2404 | 0.49s/batch | ETA 37.2 min
  [1/5] batch 250/4806 loss=1.3271 | 0.48s/batch | ETA 36.8 min
  [1/5] batch 300/4806 loss=1.2782 | 0.48s/batch | ETA 36.3 min
  [1/5] batch 350/4806 loss=1.3407 | 0.48s/batch | ETA 35.9 min
  [1/5] batch 400/4806 loss=1.2519 | 0.48s/batch | ETA 35.4 min
  [1/5] batch 450/4806 loss=1.2712 | 0.48s/batch | ETA 35.0 min
  [1/5] batch 500/4806 loss=1.2654 | 0.48s/batch | ETA 34.6 min
  [1/5] batch 550/4806 loss=1.2402 | 0.48s/batch | ETA 34.1 min
  [1/5] batch 600/4806 loss=1.3047 | 0.48s/batch | ETA 33.7 min
  [1/5] batch 650/4806 loss=1.3165 | 0.48s/batch | ETA 33.

In [ ]:
# =============================================================================
# SECTION 19: TRAINING CURVES
# =============================================================================

def plot_training_curves(ht, hs):
    keys = [("train_loss", "Train Loss"), ("val_loss",  "Val Loss"),
            ("miou",       "Val mIoU"),   ("dice",      "Val Dice"),
            ("pixel_acc",  "Val Pixel Accuracy")]
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    axes = axes.flatten()
    for ax, (k, title) in zip(axes, keys):
        ax.plot(ht[k], label="Transfer", color="#4e79a7", lw=2)
        ax.plot(hs[k], label="Scratch",  color="#e15759", lw=2, ls="--")
        ax.set_title(title); ax.set_xlabel("Epoch")
        ax.legend(); ax.grid(alpha=0.3)
    axes[5].plot(ht["lr"], color="#4e79a7", label="Transfer")
    axes[5].plot(hs["lr"], color="#e15759", label="Scratch", ls="--")
    axes[5].set_title("Learning Rate"); axes[5].set_yscale("log")
    axes[5].legend(); axes[5].grid(alpha=0.3)
    plt.suptitle("U-Net Training Curves — Transfer vs Scratch", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "unet_training_curves.png"), dpi=150)
    plt.close()
    print("Saved unet_training_curves.png")

plot_training_curves(hist_transfer, hist_scratch)


# =============================================================================
# SECTION 20: FULL EVALUATION ON HELD-OUT VAL SET
# Metrics: mIoU, Dice, pixel accuracy, F1/P/R (per-class + macro)
# =============================================================================

@torch.no_grad()
def evaluate(model, loader, model_name="model"):
    model.eval()
    preds_all, trues_all, probs_all = [], [], []

    for imgs, masks in loader:
        logits = model(imgs.to(DEVICE))               # [B, C, H, W]
        probs  = F.softmax(logits, dim=1)
        preds  = logits.argmax(1)
        for p, g, pr in zip(preds.cpu().numpy(),
                             masks.numpy(),
                             probs.cpu().numpy()):
            preds_all.append(p.flatten())
            trues_all.append(g.flatten())
            probs_all.append(pr.reshape(num_labels, -1).T)   # [H*W, C]

    P = np.concatenate(preds_all)
    T = np.concatenate(trues_all)
    PR = np.concatenate(probs_all, axis=0)             # [N_px, C]

    # per-class IoU & Dice
    iou_cls  = np.full(num_labels, np.nan)
    dice_cls = np.full(num_labels, np.nan)
    for c in range(num_labels):
        pc = P == c; tc = T == c
        inter = (pc & tc).sum(); union = (pc | tc).sum()
        denom = pc.sum() + tc.sum()
        if union > 0: iou_cls[c]  = inter / union
        if denom > 0: dice_cls[c] = (2*inter + 1) / (denom + 1)

    macro_miou = float(np.nanmean(iou_cls[1:]))
    macro_dice = float(np.nanmean(dice_cls[1:]))
    pix_acc_v  = float((P == T).mean())

    # F1 / Precision / Recall on foreground pixels
    fg       = T > 0
    labs     = list(range(1, num_labels))
    f1_mac   = f1_score(T[fg], P[fg], average="macro",  labels=labs, zero_division=0)
    prec_mac = precision_score(T[fg], P[fg], average="macro", labels=labs, zero_division=0)
    rec_mac  = recall_score(T[fg], P[fg], average="macro",  labels=labs, zero_division=0)

    # per-class F1
    f1_cls = f1_score(T[fg], P[fg], average=None, labels=labs, zero_division=0)

    print(f"\n{'='*55}")
    print(f"EVALUATION — {model_name}")
    print(f"{'='*55}")
    print(f"  mIoU (fg macro)  : {macro_miou:.4f}")
    print(f"  Dice (fg macro)  : {macro_dice:.4f}")
    print(f"  Pixel Accuracy   : {pix_acc_v:.4f}")
    print(f"  Macro F1 (fg px) : {f1_mac:.4f}")
    print(f"  Macro Prec       : {prec_mac:.4f}")
    print(f"  Macro Recall     : {rec_mac:.4f}")
    print(f"\n  {'Class':25s}  {'IoU':>7}  {'Dice':>7}  {'F1':>7}")
    print(f"  {'-'*52}")
    for i, name in enumerate(CLASS_NAMES):
        i_s = f"{iou_cls[i]:.4f}"  if not np.isnan(iou_cls[i])  else "  N/A "
        d_s = f"{dice_cls[i]:.4f}" if not np.isnan(dice_cls[i]) else "  N/A "
        f_s = f"{f1_cls[i-1]:.4f}" if i > 0 else "  N/A "
        print(f"  {name:25s}  {i_s:>7}  {d_s:>7}  {f_s:>7}")

    return dict(model_name=model_name,
                miou=macro_miou, dice=macro_dice, pixel_acc=pix_acc_v,
                f1=f1_mac, prec=prec_mac, rec=rec_mac,
                iou_cls=iou_cls, dice_cls=dice_cls, f1_cls=f1_cls,
                preds_flat=P, trues_flat=T, probs_flat=PR)


print("\nEvaluating on held-out val set...")
res_transfer = evaluate(model_transfer, val_eval_loader, "Transfer Learning")
res_scratch  = evaluate(model_scratch,  val_eval_loader, "From Scratch")


# =============================================================================
# SECTION 21: ROC / AUC  (per-class, one-vs-rest on pixels)
# =============================================================================

def plot_roc(results, suffix):
    colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in range(1, num_labels):
        tb = (results["trues_flat"] == c).astype(int)
        if tb.sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(tb, results["probs_flat"][:, c])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[c-1], lw=2,
                label=f"{CLASS_NAMES[c]}  AUC={roc_auc:.3f}")
    ax.plot([0,1], [0,1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"Per-class ROC — {results['model_name']}")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"unet_roc_{suffix}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved unet_roc_{suffix}.png")

plot_roc(res_transfer, "transfer")
plot_roc(res_scratch,  "scratch")


# =============================================================================
# SECTION 22: PER-CLASS IoU BAR CHART  (Transfer vs Scratch)
# =============================================================================

def plot_iou_comparison(rt, rs):
    x = np.arange(num_labels); w = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w/2, np.nan_to_num(rt["iou_cls"]), w,
           label="Transfer", color="#4e79a7")
    ax.bar(x + w/2, np.nan_to_num(rs["iou_cls"]), w,
           label="Scratch",  color="#e15759", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=15, ha="right")
    ax.set_ylabel("IoU"); ax.set_ylim(0, 1)
    ax.set_title("Per-class IoU: Transfer vs Scratch")
    ax.legend(); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "unet_iou_comparison.png"), dpi=150)
    plt.close()
    print("Saved unet_iou_comparison.png")

plot_iou_comparison(res_transfer, res_scratch)


# =============================================================================
# SECTION 23: QUALITATIVE PREDICTIONS
# Columns: Input Image | Ground Truth | Transfer Pred | Scratch Pred
# =============================================================================

def visualize_predictions(m_tr, m_sc, loader, n=6):
    m_tr.eval(); m_sc.eval()
    imgs_show, gt_show, tr_show, sc_show = [], [], [], []

    with torch.no_grad():
        for imgs, masks in loader:
            d = imgs.to(DEVICE)
            tr_p = m_tr(d).argmax(1).cpu().numpy()
            sc_p = m_sc(d).argmax(1).cpu().numpy()
            for img, gt, tp, sp in zip(imgs.numpy(), masks.numpy(), tr_p, sc_p):
                imgs_show.append(
                    (img.transpose(1,2,0)*255).clip(0,255).astype(np.uint8))
                gt_show.append(PALETTE[gt])
                tr_show.append(PALETTE[tp])
                sc_show.append(PALETTE[sp])
                if len(imgs_show) >= n:
                    break
            if len(imgs_show) >= n:
                break

    fig, axes = plt.subplots(n, 4, figsize=(20, n*4))
    titles = ["Image", "Ground Truth", "Transfer", "Scratch"]
    for i in range(n):
        for j, arr in enumerate([imgs_show[i], gt_show[i],
                                  tr_show[i],  sc_show[i]]):
            axes[i][j].imshow(arr)
            if i == 0:
                axes[i][j].set_title(titles[j], fontsize=11, fontweight="bold")
            axes[i][j].axis("off")

    legend = [mpatches.Patch(color=PALETTE[i]/255., label=CLASS_NAMES[i])
              for i in range(num_labels)]
    fig.legend(handles=legend, loc="lower center", ncol=num_labels, fontsize=9)
    plt.suptitle("Qualitative Segmentation Results", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "unet_qualitative.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved unet_qualitative.png")

print("\nGenerating qualitative predictions...")
visualize_predictions(model_transfer, model_scratch, val_eval_loader)


# =============================================================================
# SECTION 24: PIXEL-LEVEL CONFUSION MATRIX
# =============================================================================

def plot_confusion(results, suffix):
    cm = confusion_matrix(
        results["trues_flat"], results["preds_flat"],
        labels=list(range(num_labels)), normalize="true")
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(num_labels)); ax.set_yticks(range(num_labels))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Pixel Confusion Matrix (normalised)\n{results['model_name']}")
    for i in range(num_labels):
        for j in range(num_labels):
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i,j] > 0.5 else "black", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"unet_confusion_{suffix}.png"), dpi=150)
    plt.close()
    print(f"Saved unet_confusion_{suffix}.png")

plot_confusion(res_transfer, "transfer")
plot_confusion(res_scratch,  "scratch")


# =============================================================================
# SECTION 25: SUMMARY TABLE
# =============================================================================

def print_summary(rt, rs):
    print("\n" + "="*65)
    print("FINAL SUMMARY")
    print("="*65)
    print(f"  {'Metric':28s}  {'Transfer':>10}  {'Scratch':>10}")
    print("  " + "-"*52)
    for name, tv, sv in [
        ("mIoU (fg macro)",  rt["miou"],      rs["miou"]),
        ("Dice (fg macro)",  rt["dice"],      rs["dice"]),
        ("Pixel Accuracy",   rt["pixel_acc"], rs["pixel_acc"]),
        ("Macro F1 (fg px)", rt["f1"],        rs["f1"]),
        ("Macro Precision",  rt["prec"],      rs["prec"]),
        ("Macro Recall",     rt["rec"],       rs["rec"]),
    ]:
        print(f"  {name:28s}  {tv:>10.4f}  {sv:>10.4f}")

    print(f"\n  {'Class':25s}  {'Tr IoU':>8}  {'Sc IoU':>8}  "
          f"{'Tr Dice':>8}  {'Sc Dice':>8}  {'Tr F1':>7}  {'Sc F1':>7}")
    print("  " + "-"*79)
    for i, name in enumerate(CLASS_NAMES):
        fmt = lambda v: f"{v:.4f}" if not np.isnan(v) else "   N/A"
        ti = rt["iou_cls"][i];  si = rs["iou_cls"][i]
        td = rt["dice_cls"][i]; sd = rs["dice_cls"][i]
        tf = rt["f1_cls"][i-1] if i > 0 else float("nan")
        sf = rs["f1_cls"][i-1] if i > 0 else float("nan")
        print(f"  {name:25s}  {fmt(ti):>8}  {fmt(si):>8}  "
              f"{fmt(td):>8}  {fmt(sd):>8}  {fmt(tf):>7}  {fmt(sf):>7}")
    print("="*65)

print_summary(res_transfer, res_scratch)


# =============================================================================
# SECTION 26: SAVE MODELS + CONFIG
# =============================================================================

torch.save(model_transfer.state_dict(),
           os.path.join(OUTPUT_DIR, "unet_transfer_final.pth"))
torch.save(model_scratch.state_dict(),
           os.path.join(OUTPUT_DIR, "unet_scratch_final.pth"))

config = {
    "model":          "unet_resnet34",
    "num_labels":     num_labels,
    "img_size":       IMG_SIZE,
    "class_names":    CLASS_NAMES,
    "cat_to_label":   {str(k): v for k, v in cat_to_label.items()},
    "index_to_cat":   {str(k): v for k, v in index_to_cat.items()},
    "subsample_step": SUBSAMPLE_STEP,
}
with open(os.path.join(OUTPUT_DIR, "unet_config.json"), "w") as f:
    json.dump(config, f, indent=4)

print("\nSaved: unet_transfer_final.pth  unet_scratch_final.pth  unet_config.json")
print("\nAll output files:")
for fname in [
    "unet_transfer_final.pth", "unet_scratch_final.pth",
    "unet_transfer_best.pth",  "unet_scratch_best.pth",
    "unet_config.json",        "label_map.json",
    "unet_distribution.png",   "unet_augmentation_preview.png",
    "unet_train_batch.png",    "unet_training_curves.png",
    "unet_roc_transfer.png",   "unet_roc_scratch.png",
    "unet_iou_comparison.png", "unet_qualitative.png",
    "unet_confusion_transfer.png", "unet_confusion_scratch.png",
]:
    print(f"  {fname}")


# =============================================================================
# SECTION 27: INFERENCE FUNCTION
# =============================================================================

def run_inference(model_path, image_paths, config_path=None, device=None):
    """
    Args:
        model_path  : path to saved .pth state dict
        image_paths : list of image file paths
        config_path : optional path to unet_config.json
        device      : torch.device (auto-detected if None)

    Returns:
        list of dicts, one per image:
            pred_mask   : np.ndarray [H, W] int64   class indices 0-5
            prob_map    : np.ndarray [C, H, W] float32  softmax probs
            label_names : np.ndarray [H, W] str     class name per pixel
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if config_path and os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
        img_size  = cfg["img_size"]
        n_labels  = cfg["num_labels"]
        cls_names = cfg["class_names"]
    else:
        img_size, n_labels, cls_names = IMG_SIZE, num_labels, CLASS_NAMES

    model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                     in_channels=3, classes=n_labels, activation=None).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    name_arr = np.array(cls_names)
    results  = []
    with torch.no_grad():
        for path in image_paths:
            img   = Image.open(path).convert("RGB").resize((img_size, img_size))
            img_t = TF.to_tensor(img).unsqueeze(0).to(device)
            logits = model(img_t)[0]                       # [C, H, W]
            probs  = F.softmax(logits, dim=0).cpu().numpy().astype(np.float32)
            pred   = probs.argmax(0).astype(np.int64)
            results.append({
                "pred_mask":   pred,
                "prob_map":    probs,
                "label_names": name_arr[pred],
            })
    return results


print("\n✓ All done. Outputs saved to", OUTPUT_DIR)

Saved unet_training_curves.png

Evaluating on held-out val set...
